# Spark Setup

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("TimeSeriesAnalysis_amritap1")
    .config("spark.driver.memory", "10g")
    .config("spark.executor.memory", "10g")
    .config("spark.executor.cores", "1")
    .config("spark.executor.instances", "14")
    .config("spark.serializer","org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryoserializer.buffer","1g")
    .config("spark.driver.maxResultSize","8g")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/12 06:34:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/12 06:34:47 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/12/12 06:34:47 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


# Data Cleaning

In [2]:
from pyspark.sql import functions as F

# Load data
commentsDF = spark.read.parquet("/project/macs40123/amritap1/macs-40123-amritapathak1/final_data_with_id/comments")
submissionsDF = spark.read.parquet("/project/macs40123/amritap1/macs-40123-amritapathak1/final_data_with_id/submissions")

In [3]:
print(f"Comments: {commentsDF.count()}")
print(f"Submissions: {submissionsDF.count()}")

Comments: 62790822
Submissions: 1955840


In [5]:
# Clean and prepare data with timestamps
def clean_time_series_data(df, text_column):
    df = df.filter(F.col(text_column).isNotNull())
    df = df.filter(~F.col(text_column).isin("[deleted]", "[removed]"))
    
    # Remove bots
    a = F.lower(F.col("author"))
    is_bot = F.coalesce(
        a.contains("automoderator") | a.contains("moderator") | a.contains("bot") | a.contains("automod"),
        F.lit(False)
    )
    df = df.filter(~is_bot)
    
    # Convert timestamps
    df = df.withColumn(
        "created_ts",
        F.to_timestamp(F.col("created_utc"))
    )
    
    return df

# Apply cleaning
commentsDF_clean = clean_time_series_data(commentsDF, "body")
submissionsDF_clean = clean_time_series_data(submissionsDF, "selftext")

In [6]:
# Filter out records with NULL timestamps
commentsDF_clean = commentsDF_clean.filter(F.col("created_ts").isNotNull())
submissionsDF_clean = submissionsDF_clean.filter(F.col("created_ts").isNotNull())

print(f"Comments after filtering: {commentsDF_clean.count()}")
print(f"Submissions after filtering: {submissionsDF_clean.count()}")

Comments after filtering: 51065749


[Stage 11:==================================================>       (7 + 1) / 8]

Submissions after filtering: 659460


In [7]:
print(f"Submissions after filtering: {submissionsDF_clean.filter(F.col("num_comments").isNotNull()).count()}")

[Stage 14:====================================>                     (5 + 3) / 8]

Submissions after filtering: 659460


## Time Series Aggregation

In [8]:
# Aggregate by day
daily_activity = (submissionsDF_clean
    .withColumn("date", F.to_date("created_ts"))
    .groupBy("date")
    .agg(
        F.count("*").alias("submission_count"),
        F.sum("num_comments").alias("total_comments"),
        F.avg("num_comments").alias("avg_comments_per_post"),
        F.avg("score").alias("avg_score")
    )
    .orderBy("date")
)

daily_activity.show(20)

+----------+----------------+--------------+---------------------+------------------+
|      date|submission_count|total_comments|avg_comments_per_post|         avg_score|
+----------+----------------+--------------+---------------------+------------------+
|2019-12-31|              18|           288|                 16.0|             181.0|
|2020-01-01|              66|          1902|   28.818181818181817|             156.0|
|2020-01-02|             172|          4422|   25.709302325581394|126.16279069767442|
|2020-01-03|             162|          9279|    57.27777777777778|183.33333333333334|
|2020-01-04|              63|          1495|    23.73015873015873|385.76190476190476|
|2020-01-05|              67|          2273|    33.92537313432836|412.92537313432837|
|2020-01-06|             106|          2708|   25.547169811320753|180.53773584905662|
|2020-01-07|             177|          4840|    27.34463276836158|  368.225988700565|
|2020-01-08|             140|          3625|   25.8928

In [9]:
# Weekly aggregation
weekly_activity = (daily_activity
    .withColumn("week", F.weekofyear("date"))
    .withColumn("year", F.year("date"))
    .groupBy("year", "week")
    .agg(
        F.sum("submission_count").alias("weekly_submissions"),
        F.sum("total_comments").alias("weekly_comments"),
        F.avg("avg_score").alias("avg_weekly_score")
    )
    .orderBy("year", "week")
)

# Monthly aggregation
monthly_activity = (daily_activity
    .withColumn("month", F.month("date"))
    .withColumn("year", F.year("date"))
    .groupBy("year", "month")
    .agg(
        F.sum("submission_count").alias("monthly_submissions"),
        F.sum("total_comments").alias("monthly_comments"),
        F.avg("avg_score").alias("avg_monthly_score")
    )
    .orderBy("year", "month")
)

monthly_activity.show(20)

+----+-----+-------------------+----------------+------------------+
|year|month|monthly_submissions|monthly_comments| avg_monthly_score|
+----+-----+-------------------+----------------+------------------+
|2019|   12|                 18|             288|             181.0|
|2020|    1|               4578|          173481| 175.9644000137804|
|2020|    2|               9157|          340643|209.54564931131296|
|2020|    3|              17265|          833153|179.97844664824837|
|2020|    4|              13139|          630298|217.72592485476517|
|2020|    5|               8432|          293005|  312.078003008974|
|2020|    6|               7723|          225179|281.96630453368647|
|2020|    7|               7146|          256303| 228.4196626902647|
|2020|    8|               7551|          287488|212.77250296055843|
|2020|    9|               8482|          213407|292.73105070023604|
|2020|   10|               6595|          186330| 275.1471131730648|
|2020|   11|              12140|  

## Moving Averages

This allows us to:
- Reduce daily volatility by smoothing out single day variations
- Reveal underlying trends by focusing on activity over time
- Filter out the noise in the data by averaging out the random fluctuations

In [10]:
from pyspark.sql.window import Window

# Define window specifications for moving averages
window_7 = Window.orderBy("date").rowsBetween(-6, 0)
window_30 = Window.orderBy("date").rowsBetween(-29, 0)

# Calculate moving averages
daily_with_ma = (daily_activity
    .withColumn("ma_7_submissions", F.avg("submission_count").over(window_7))
    .withColumn("ma_30_submissions", F.avg("submission_count").over(window_30))
    .withColumn("ma_7_comments", F.avg("total_comments").over(window_7))
    .withColumn("ma_30_comments", F.avg("total_comments").over(window_30))
)

daily_with_ma.select("date", "submission_count", "total_comments", "ma_7_submissions", "ma_7_comments").show(20)

25/12/12 06:40:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 0

+----------+----------------+--------------+------------------+------------------+
|      date|submission_count|total_comments|  ma_7_submissions|     ma_7_comments|
+----------+----------------+--------------+------------------+------------------+
|2019-12-31|              18|           288|              18.0|             288.0|
|2020-01-01|              66|          1902|              42.0|            1095.0|
|2020-01-02|             172|          4422| 85.33333333333333|            2204.0|
|2020-01-03|             162|          9279|             104.5|           3972.75|
|2020-01-04|              63|          1495|              96.2|            3477.2|
|2020-01-05|              67|          2273| 91.33333333333333|            3276.5|
|2020-01-06|             106|          2708| 93.42857142857143| 3195.285714285714|
|2020-01-07|             177|          4840|116.14285714285714|3845.5714285714284|
|2020-01-08|             140|          3625|126.71428571428571| 4091.714285714286|
|202

## Exponential Moving Averages

Consider the moving averages but with more weight given to the recent observations. With alpha 0.3, 30% of the weight goes to current day's value, while 70% goes to previous EMA (exponential moving average) value. This gives more influence to the recent data.

In [11]:
# Exponential smoothing with alpha = 0.3
alpha = 0.3

# Collect data to compute exponentially smoothed values
daily_pandas = daily_activity.select("date", "submission_count", "total_comments").orderBy("date").toPandas()

# Apply exponential smoothing
daily_pandas['ema_submissions'] = daily_pandas['submission_count'].ewm(alpha=alpha, adjust=False).mean()
daily_pandas['ema_comments'] = daily_pandas['total_comments'].ewm(alpha=alpha, adjust=False).mean()

# Convert back to Spark DataFrame
daily_with_ema = spark.createDataFrame(daily_pandas)
daily_with_ema.show(20)

[Stage 52:>                                                         (0 + 1) / 1]

+----------+----------------+--------------+------------------+------------------+
|      date|submission_count|total_comments|   ema_submissions|      ema_comments|
+----------+----------------+--------------+------------------+------------------+
|2019-12-31|              18|           288|              18.0|             288.0|
|2020-01-01|              66|          1902|              32.4|             772.2|
|2020-01-02|             172|          4422|             74.28|1867.1399999999999|
|2020-01-03|             162|          9279|           100.596|4090.6979999999994|
|2020-01-04|              63|          1495| 89.31719999999999| 3311.988599999999|
|2020-01-05|              67|          2273| 82.62203999999998|3000.2920199999994|
|2020-01-06|             106|          2708| 89.63542799999999|2912.6044139999995|
|2020-01-07|             177|          4840|115.84479959999999|3490.8230897999993|
|2020-01-08|             140|          3625|123.09135971999999|3531.0761628599994|
|202

## Trend Detection and Decomposition

Analyse the long-term patters in the data

### Linear Trend Detection

Use a linear regression model to identify the overall trend

In [12]:
# Linear trend using window functions for submissions
window_spec = Window.orderBy("date")

trend_data = (daily_activity
    .withColumn("day_index", F.row_number().over(window_spec))
    .withColumn("submission_count_double", F.col("submission_count").cast("double"))
)

# Calculate linear trend coefficients
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

assembler = VectorAssembler(inputCols=["day_index"], outputCol="features")
trend_features = assembler.transform(trend_data)

lr = LinearRegression(featuresCol="features", labelCol="submission_count_double")
lr_model = lr.fit(trend_features)

print(f"Submission Trend coefficient (slope): {lr_model.coefficients[0]:.4f}")
print(f"Intercept: {lr_model.intercept:.4f}")

# Add trend predictions
trend_predictions = lr_model.transform(trend_features)
trend_predictions.select("date", "submission_count", "total_comments", "prediction").show(20)

25/12/12 06:40:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 0

Submission Trend coefficient (slope): -0.3730
Intercept: 805.8981


25/12/12 06:40:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 0

+----------+----------------+--------------+-----------------+
|      date|submission_count|total_comments|       prediction|
+----------+----------------+--------------+-----------------+
|2019-12-31|              18|           288|805.5251978819533|
|2020-01-01|              66|          1902|805.1522478621972|
|2020-01-02|             172|          4422|804.7792978424411|
|2020-01-03|             162|          9279| 804.406347822685|
|2020-01-04|              63|          1495|804.0333978029289|
|2020-01-05|              67|          2273|803.6604477831728|
|2020-01-06|             106|          2708|803.2874977634167|
|2020-01-07|             177|          4840|802.9145477436606|
|2020-01-08|             140|          3625|802.5415977239045|
|2020-01-09|             166|          3836|802.1686477041484|
|2020-01-10|             109|          9575|801.7956976843923|
|2020-01-11|              80|          2475|801.4227476646362|
|2020-01-12|              70|          3546|801.0497976

The positive slope (0.1099) indicates that the subreddit is growing at around ~0.11 submissions per day, which would also translate to around 0.11 * 365 ≈ 40 more submissions annually

The negative intercept (-15.7653) predicts negative submissions on day 0 (December 31st, 2013). This indicates that the model underpredicts the activity in the early days, which can be explained by the high variance in the data (ranging from 1-12 submissions).

A linear model might be too simplistic for the data. One potential solution could be to use a polynomial or exponential model to better capture the complex nature of the data.

### Seasonality Detection

Detecting recurring patterns on different time scales.

In [13]:
# Detect seasonality by day of week
daily_with_dow = (daily_activity
    .withColumn("day_of_week", F.dayofweek("date"))
    .withColumn("month", F.month("date"))
)

# Average activity by day of week
dow_patterns = (daily_with_dow
    .groupBy("day_of_week")
    .agg(
        F.avg("submission_count").alias("avg_submissions"),
        F.avg("total_comments").alias("avg_comments")
    )
    .orderBy("day_of_week")
)

dow_patterns.show()

# Monthly seasonal patterns
monthly_patterns = (daily_with_dow
    .groupBy("month")
    .agg(
        F.avg("submission_count").alias("avg_submissions"),
        F.avg("total_comments").alias("avg_comments")
    )
    .orderBy("month")
)

monthly_patterns.show()

+-----------+------------------+------------------+
|day_of_week|   avg_submissions|      avg_comments|
+-----------+------------------+------------------+
|          1| 308.3269230769231|15364.929487179486|
|          2|  581.974358974359|  32469.1858974359|
|          3|  660.343949044586|35880.458598726116|
|          4| 749.3694267515923|40165.547770700636|
|          5| 920.8726114649681|42426.019108280256|
|          6|  664.796178343949| 42263.90445859873|
|          7|320.36942675159236|12175.305732484076|
+-----------+------------------+------------------+



[Stage 103:===================================>                     (5 + 3) / 8]

+-----+------------------+------------------+
|month|   avg_submissions|      avg_comments|
+-----+------------------+------------------+
|    1|1720.8387096774193|56804.397849462366|
|    2|1562.0117647058823| 62545.32941176471|
|    3| 764.9569892473119| 51580.95698924731|
|    4|440.74444444444447|27202.944444444445|
|    5| 356.9569892473118| 19565.25806451613|
|    6| 524.3666666666667|27677.966666666667|
|    7| 273.1720430107527| 19424.78494623656|
|    8|392.40860215053766|27714.279569892475|
|    9|             301.6|23489.755555555555|
|   10|300.80645161290323| 21441.41935483871|
|   11| 342.8666666666667|23810.855555555554|
|   12| 293.3723404255319| 19352.17021276596|
+-----+------------------+------------------+



Weekly Patterns:
- Tuesday to Thursday (3-5) have the highest activity in the week, and Monday and Friday (2 and 6) also show fairly strong activity
- Weekends (1 and 7) show low activity, which is expected due to closure of markets.


Monthly Patterns:
- There's a strong surge in the spring (April and May), which could be attributed to various factors like the end of tax season and Q1 earnings reports
- June shows a sharp decline, a classic phenomenon in trading known as the summer slowdown, which then slowly recovers over July and August
- There's moderately stable activity in the Fall, showcasing normal trading patterns
- Winters show the lowest activity, with December having the lowest numbers due to the holiday season. Q1 follows the low activity and acts as the recovery period, while also preparing for the transition to the Q1 financial reports coming out in the Spring.

## Autoregression and Lag Features

Autoregression allows us to use past activity patterns to help predict the future activity. Values from previous time periods, called lag features, are used. The use of these lag features reveal how the momentum gets carried through over short or long term trends.

In [14]:
# Create lag features for submissions
window_lag = Window.orderBy("date")

lagged_data = (daily_activity
    .withColumn("lag_1", F.lag("submission_count", 1).over(window_lag))
    .withColumn("lag_7", F.lag("submission_count", 7).over(window_lag))
    .withColumn("lag_30", F.lag("submission_count", 30).over(window_lag))
    .dropna()
)

lagged_data.select("date", "submission_count", "lag_1", "lag_7", "lag_30").show(20)

25/12/12 06:40:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 0

+----------+----------------+-----+-----+------+
|      date|submission_count|lag_1|lag_7|lag_30|
+----------+----------------+-----+-----+------+
|2020-01-30|             224|  230|  161|    18|
|2020-01-31|             229|  224|  243|    66|
|2020-02-01|             125|  229|  112|   172|
|2020-02-02|             166|  125|   74|   162|
|2020-02-03|             272|  166|  178|    63|
|2020-02-04|             515|  272|  173|    67|
|2020-02-05|             341|  515|  230|   106|
|2020-02-06|             283|  341|  224|   177|
|2020-02-07|             222|  283|  229|   140|
|2020-02-08|             128|  222|  125|   166|
|2020-02-09|             121|  128|  166|   109|
|2020-02-10|             285|  121|  272|    80|
|2020-02-11|             297|  285|  515|    70|
|2020-02-12|             312|  297|  341|   201|
|2020-02-13|             257|  312|  283|   182|
|2020-02-14|             270|  257|  222|   188|
|2020-02-15|             124|  270|  128|   216|
|2020-02-16|        

In [15]:
# Fit autoregressive model for submissions
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

assembler_ar = VectorAssembler(
    inputCols=["lag_1", "lag_7", "lag_30"], 
    outputCol="features"
)

ar_features = assembler_ar.transform(lagged_data)

lr_ar = LinearRegression(featuresCol="features", labelCol="submission_count")
ar_model = lr_ar.fit(ar_features)

print(f"AR Model Coefficients: {ar_model.coefficients}")
print(f"AR Model RMSE: {ar_model.summary.rootMeanSquaredError:.2f}")
print(f"AR Model R2: {ar_model.summary.r2:.4f}")

25/12/12 06:40:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 0

AR Model Coefficients: [0.7855079520378548,0.012729291585622268,-0.010685413177892053]
AR Model RMSE: 1333.01
AR Model R2: 0.6234


25/12/12 06:40:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


The coefficients for short-term (yesterday), mid-term (weekly), and long-term (monthly) lag features are 0.514, 0.549 and -0.146 respectively.

This indicates that the short to mid term features have the strongest positive influence on predicting the current day's activity. Whereas, the long term features have a negative coefficient, which indicates a mean reversion effect. Thus, high activity in the 30 day lag feature translates to low activity in the current day, and vice versa, which captures the fluctuating nature of the activity around financial discourse

RMSE of 15.66 and R2 of 82.84% also indicate a strong fit of the model on the data.

## Change Point Detection

Detecting time periods with anomalous activity using z-scores. These could be due to various viral events like market crashes or meme stock movements, political or financial surprises in major firms, or collective movements.

In [16]:
# Calculate rolling statistics for change detection (using submissions)
window_60 = Window.orderBy("date").rowsBetween(-59, 0)

change_detection = (daily_activity
    .withColumn("rolling_mean", F.avg("submission_count").over(window_60))
    .withColumn("rolling_std", F.stddev("submission_count").over(window_60))
    .withColumn("z_score", 
                (F.col("submission_count") - F.col("rolling_mean")) / F.col("rolling_std"))
)

# Identify anomalies (z-score > 3)
anomalies = change_detection.filter(F.abs(F.col("z_score")) > 3)
print(f"Number of detected anomalies: {anomalies.count()}")
anomalies.select("date", "submission_count", "total_comments", "rolling_mean", "z_score").show(20)

25/12/12 06:40:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 0

Number of detected anomalies: 26


25/12/12 06:40:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:40:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 0

+----------+----------------+--------------+------------------+------------------+
|      date|submission_count|total_comments|      rolling_mean|           z_score|
+----------+----------------+--------------+------------------+------------------+
|2020-02-04|             515|         13686|157.61111111111111|4.1358997341307004|
|2020-02-27|             814|         15532|213.57627118644066| 4.272236287155851|
|2020-02-28|             769|         54356|222.83333333333334|3.4851690433348548|
|2020-03-09|             869|         21791|278.43333333333334| 3.279729672990053|
|2020-03-12|            1175|         27378|             311.3|  4.05817253434978|
|2020-09-03|             807|         16711|            260.35|4.7387353673761305|
|2020-11-23|             609|         15398|241.66666666666666| 3.098429973223632|
|2020-11-24|             847|         19132|252.86666666666667| 4.194559519864329|
|2020-11-25|            1155|         18552|270.51666666666665| 4.859866941228536|
|202

## Stock Ticker Trends Over Time

Trends for popular stock tickers like meme stocks (GME and AMC), tech giants (like MSFT and NVDA), and market indices (like SPY and QQQ)

In [17]:
# Extract ticker mentions from submission titles and text
from pyspark.sql import types as T

tickers = ["AAPL", "TSLA", "NVDA", "GME", "AMC", "MSFT", "SPY", "QQQ"]
ticker_set = set(tickers)

@F.udf(T.ArrayType(T.StringType()))
def extract_tickers(text):
    if not text:
        return []
    text_upper = text.upper()
    found = [t for t in ticker_set if t in text_upper]
    return found

# Apply to submissions (title + selftext)
submissions_with_tickers = (submissionsDF_clean
    .withColumn("date", F.to_date("created_ts"))
    .withColumn("combined_text", F.concat_ws(" ", F.col("title"), F.coalesce(F.col("selftext"), F.lit(""))))
    .withColumn("tickers", extract_tickers(F.col("combined_text")))
    .filter(F.size("tickers") > 0)
    .withColumn("ticker", F.explode("tickers"))
)

# Count ticker mentions by day
ticker_time_series = (submissions_with_tickers
    .groupBy("date", "ticker")
    .agg(F.count("*").alias("mention_count"))
    .orderBy("date", "ticker")
)

ticker_time_series.show(30)

[Stage 177:==========================================>              (6 + 2) / 8]

+----------+------+-------------+
|      date|ticker|mention_count|
+----------+------+-------------+
|2019-12-31|   GME|            1|
|2019-12-31|   SPY|            1|
|2020-01-01|  MSFT|            1|
|2020-01-01|   SPY|            5|
|2020-01-01|  TSLA|            4|
|2020-01-02|  AAPL|            2|
|2020-01-02|  MSFT|            3|
|2020-01-02|  NVDA|            1|
|2020-01-02|   SPY|            7|
|2020-01-02|  TSLA|            5|
|2020-01-03|  AAPL|            2|
|2020-01-03|  MSFT|            1|
|2020-01-03|   SPY|            8|
|2020-01-03|  TSLA|            6|
|2020-01-04|  AAPL|            2|
|2020-01-04|  MSFT|            1|
|2020-01-04|   QQQ|            1|
|2020-01-04|   SPY|            2|
|2020-01-05|  NVDA|            1|
|2020-01-05|   SPY|            6|
|2020-01-05|  TSLA|            1|
|2020-01-06|  AAPL|            2|
|2020-01-06|  NVDA|            1|
|2020-01-06|   SPY|            5|
|2020-01-06|  TSLA|            5|
|2020-01-07|  AAPL|            1|
|2020-01-07|  

In [18]:
# Moving average for individual tickers
ticker_window = Window.partitionBy("ticker").orderBy("date").rowsBetween(-6, 0)

ticker_smoothed = (ticker_time_series
    .withColumn("ma_7", F.avg("mention_count").over(ticker_window))
)

# Show trends for GME
ticker_smoothed.filter(F.col("ticker") == "GME").show(30)

+----------+------+-------------+------------------+
|      date|ticker|mention_count|              ma_7|
+----------+------+-------------+------------------+
|2019-12-31|   GME|            1|               1.0|
|2020-01-13|   GME|            2|               1.5|
|2020-01-14|   GME|            4|2.3333333333333335|
|2020-01-15|   GME|            2|              2.25|
|2020-01-17|   GME|            1|               2.0|
|2020-01-19|   GME|            2|               2.0|
|2020-01-23|   GME|            2|               2.0|
|2020-01-24|   GME|            2| 2.142857142857143|
|2020-01-26|   GME|            1|               2.0|
|2020-01-27|   GME|            2|1.7142857142857142|
|2020-01-29|   GME|            1|1.5714285714285714|
|2020-01-31|   GME|            2|1.7142857142857142|
|2020-02-01|   GME|            1|1.5714285714285714|
|2020-02-02|   GME|            1|1.4285714285714286|
|2020-02-04|   GME|            1|1.2857142857142858|
|2020-02-06|   GME|            1|1.28571428571

## Cross-Correlation Analysis

Analysis of the relationship between different activity metrics like number of submissions, number of comments, average score of the submissions, etc

In [19]:
# Compute correlation between submissions and total comments
correlation_data = daily_activity.select("submission_count", "total_comments", "avg_score").toPandas()
correlation = correlation_data.corr()
print("Correlation Matrix:")
print(correlation)

# Lagged correlation - does submission count predict future comments?
lagged_corr_data = (daily_activity
    .withColumn("submissions_lag_1", F.lag("submission_count", 1).over(window_lag))
    .withColumn("comments_lag_1", F.lag("total_comments", 1).over(window_lag))
    .dropna()
    .select("submission_count", "total_comments", "submissions_lag_1", "comments_lag_1")
    .toPandas()
)

print("\nCorrelation with lagged values:")
print(lagged_corr_data.corr())

Correlation Matrix:
                  submission_count  total_comments  avg_score
submission_count          1.000000        0.917417  -0.059781
total_comments            0.917417        1.000000  -0.072530
avg_score                -0.059781       -0.072530   1.000000


25/12/12 06:41:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:41:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:41:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:41:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:41:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 06:41:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/12 0


Correlation with lagged values:
                   submission_count  total_comments  submissions_lag_1  \
submission_count           1.000000        0.917424           0.789646   
total_comments             0.917424        1.000000           0.733789   
submissions_lag_1          0.789646        0.733789           1.000000   
comments_lag_1             0.788157        0.822286           0.917423   

                   comments_lag_1  
submission_count         0.788157  
total_comments           0.822286  
submissions_lag_1        0.917423  
comments_lag_1           1.000000  


Same Day Correlations:
- There's a high positive correlation between submissions count and total comments, which is as expected, as more posts lead to more discussions.
- There's a strong correlation between total number of comments and the highest average score for a post too, which indicates that the quality of the posts drive engagement, both in terms of upvotes and discussions.
- There's a moderate correlation between the number of posts and the average score of each post. This is slightly surprising, as it indicates that days with high volume of posts do not lead to the diluted quality of these posts.

Lagged Correlations:
- There is a strong correlation between the current day number of submissions and the day 1 lag number of submissions, which confirms the Autoregression model's predictions that activity carries over momentum over short periods of time.
- There is also strong correlation between number of comments and number of submissions with day 1 lag, implying that the posts need time to accumulate comments and discussions.
- Strong correlation between day 1 lag's number of comments and current day's number of submissions also hints towards an existence of a feedback loop - that active discussions lead to new posts in the short term

## Summary Statistics and Insights

Understanding the trajectory of growth of the subreddit's activity over the considered time period.

In [20]:
# Overall time series statistics
print("=== Time Series Summary ===")
print(f"Total days analyzed: {daily_activity.count()}")
print(f"Date range: {daily_activity.agg(F.min('date')).collect()[0][0]} to {daily_activity.agg(F.max('date')).collect()[0][0]}")

# Activity statistics
activity_stats = daily_activity.select(
    F.mean("submission_count").alias("avg_daily_submissions"),
    F.stddev("submission_count").alias("std_daily_submissions"),
    F.max("submission_count").alias("max_daily_submissions"),
    F.mean("total_comments").alias("avg_daily_comments"),
    F.stddev("total_comments").alias("std_daily_comments"),
    F.max("total_comments").alias("max_daily_comments"),
    F.mean("avg_comments_per_post").alias("avg_comments_per_post")
).collect()[0]

print(f"\nAverage daily submissions: {activity_stats['avg_daily_submissions']:.2f}")
print(f"Std dev daily submissions: {activity_stats['std_daily_submissions']:.2f}")
print(f"Max daily submissions: {activity_stats['max_daily_submissions']}")

print(f"\nAverage daily comments (total): {activity_stats['avg_daily_comments']:.2f}")
print(f"Average comments per post: {activity_stats['avg_comments_per_post']:.2f}")

print(f"Std dev daily comments: {activity_stats['std_daily_comments']:.2f}")
print(f"Max daily comments: {activity_stats['max_daily_comments']}")

=== Time Series Summary ===


Total days analyzed: 1097


Date range: 2019-12-31 to 2022-12-31



Average daily submissions: 601.15
Std dev daily submissions: 2144.59
Max daily submissions: 49038

Average daily comments (total): 31548.94
Average comments per post: 61.85
Std dev daily comments: 66149.88
Max daily comments: 1252788


The standard deviation is higher than the mean for submissions, indicating towards a highly volatile community in terms of activity. Due to the event-driven nature of the financial discourse, this is aligned with the expected behavior.

In [21]:
# Year-over-year growth
yearly_stats = (daily_activity
    .withColumn("year", F.year("date"))
    .groupBy("year")
    .agg(
        F.sum("submission_count").alias("total_submissions"),
        F.sum("total_comments").alias("total_comments")
    )
    .orderBy("year")
)

print("\n=== Year-over-Year Activity ===")
yearly_stats.show()

# Calculate growth rates
yearly_pandas = yearly_stats.toPandas()
yearly_pandas['submission_growth'] = yearly_pandas['total_submissions'].pct_change() * 100
yearly_pandas['comment_growth'] = yearly_pandas['total_comments'].pct_change() * 100

print("\nYear-over-Year Growth Rates (%):")
print(yearly_pandas[['year', 'submission_growth', 'comment_growth']])


=== Year-over-Year Activity ===


+----+-----------------+--------------+
|year|total_submissions|total_comments|
+----+-----------------+--------------+
|2019|               18|           288|
|2020|           113943|       4323093|
|2021|           431667|      22271278|
|2022|           113832|       8014527|
+----+-----------------+--------------+




Year-over-Year Growth Rates (%):
   year  submission_growth  comment_growth
0  2019                NaN             NaN
1  2020      632916.666667    1.500974e+06
2  2021         278.844686    4.151700e+02
3  2022         -73.629673   -6.401407e+01


The growth of total comments per submission across the years indicate that the users, over time, are having deeper discussions surrounding topics. This data is also from the pre-mainstream era of the subreddit (prior to the GME short squeeze days), and thus shows the maturity and popularity of the subreddit.

The analysis on growth hints towards a quick increase in the subreddit's activity going from 2014 to 2015. The anomalous growth rates and activity can be explained by the fact that 2014 was the start year in the data (with one additional day in 2013), and there is a large chunk of missing data from 2016-2018, which would further help analyse long term patterns.